<a href="https://colab.research.google.com/github/carlosjimenez88M/ai_engineering_henry/blob/main/02-vector_data_bases/00_clase_de_repaso.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MIT Applied AI Engineering: Clase de Repaso Exhaustiva (Módulos 01 y 02)

---

**Profesor:** El rigor académico y práctico no permite saltos de fe. Me han pedido un repaso absoluto, así que repasaremos **absolutamente todo** lo que se ha visto a lo largo de las 24+ lecciones de los dos primeros módulos de Ingeniería en Inteligencia Artificial. No dejaremos ni una sola sub-carpeta sin revisar. Esta libreta condensa de verdad todo el código, desde las bases de Python hasta la orquestación Agentic RAG de Batman.

Asegúrense de tener las variables de entorno `.env` en orden. Si en esta celda introductoria falta algo, corran la instalación inicial.

In [1]:
# Instalación Absoluta de todo lo visto y por venir
# %pip install -qU langchain langchain-openai langchain-community langgraph chromadb pydantic tiktoken python-dotenv rank_bm25 scikit-learn transformers sentence-transformers faiss-cpu fastapi uvicorn pytest

import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    print("[ALERTA MIT] OPENAI_API_KEY requerida. Varias celdas de inferencia a continuación fallarán si no proveen el token de acceso.")


---
## Módulo 01: Las bases de Software Engineering en Python (`00_python_extra_class` y `01_introduction`)

Antes de conectarnos con `LangChain`, un AI Engineer sigue siendo, fundamentally, un Software Engineer.
1. **OOP (Object-Oriented Programming):** Entendimos cómo modelar agentes como clases reales y separadas de la instanciación LLM.
2. **Ejercicios Leetcode y Robustez:** Depuramos algoritmos con `O(N)` y aseguramos salidas.
3. **Pydantic:** Cuando el LLM escupe texto libre (estocástico), `Pydantic` lo amarra en tipos estrictos (determinístico).

In [2]:
from pydantic import BaseModel, Field, ValidationError
import typing

# Reflejo de `01_robustez_y_validacion.ipynb` y `01_pydantic_pipeline_ai.ipynb`
class DecisiónDeAgente(BaseModel):
    razonamiento: str = Field(description="Justificación del agente, max 500 chars", max_length=500)
    herramienta: str = Field(description="El nombre de la Tool que usaremos (ej: BusquedaDeCripto)")
    confianza: float = Field(ge=0.0, le=1.0, description="Qué tan seguro está el modelo de 0 a 1")

try:
    # Imaginemos que JSON.loads() deserializó esta salida arrojada por GPT-4
    llm_output = {
        "razonamiento": "El usuario busca precio histórico, invoco el ledger",
        "herramienta": "LedgerAPI",
        "confianza": 1.5 # MAL. Pasó el threshold lógico del mundo real.
    }
    decision = DecisiónDeAgente(**llm_output)
except ValidationError as e:
    print("¡Error de validación atrapado como en Producción!")
    print("El LLM alucinó una confianza mayor a 1.0. Esto evita que crashee nuestro backend.")
    print("Detalles Pydantic:", e.errors()[0]['msg'])

¡Error de validación atrapado como en Producción!
El LLM alucinó una confianza mayor a 1.0. Esto evita que crashee nuestro backend.
Detalles Pydantic: Input should be less than or equal to 1


---
## Módulo 01: Prompt Engineering y LCEL (`02_prompting`)

El input crudo gobierna el output stocrástico. Si envías basura, recibes basura.

- **Zero/Few Shot (`01-basic_example.ipynb`):** Darle contexto en seco vs inyectar ejemplos `input -> output` en el prompt para condicionar la distribución.
- **Rutas (Routing) Básicas (`03_routing`):** Clasificar un string para decidir qué base de código ejecutar.
- **LangChain LCEL:** Orquestación usando el comando Pipe `|` (`Prompt | Modelo | Parser`).

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Instancia Predictiva Universal
llm_mini = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# FEW SHOT Promtping - Clase 01-Introduction
prompt_extractor = ChatPromptTemplate.from_messages([
    ("system", "Extrae SOLAMENTE el nombre del Framework Tecnológico del input."),
    ("user", "Acabo de levantar el frontend con Next.js y Vercel."),
    ("assistant", "Next.js"),
    ("user", "Levanté grafos estocásticos cíclicos."),
    ("assistant", "LangGraph"),
    ("user", "{texto_usuario}")
])

# Pipeline Funcional con LCEL
extractor_chain = prompt_extractor | llm_mini | StrOutputParser()
print("---- Resultado del Few Shot Pipeline ----")
print(extractor_chain.invoke({"texto_usuario": "Estoy indexando con TF-IDF usando scikit-learn."}))

---- Resultado del Few Shot Pipeline ----


scikit-learn


## Razonamiento Complejo: CoT y ReAct (`04_cot` y `05_react` explícitos)

En lugar de forzar al modelo a dar una salida inmediata (comúnmente comete fallas matemáticas o lógicas de correlación), le pedimos que genere *Chain of Thought* (Pensamiento en Cascada).

**ReAct (Reason + Act)** lleva este Thought a la realidad combinándolo con Actuaciones externas, seguido de Observaciones `(Thought -> Action -> Observation -> Final Answer)`.

Aquí, pedimos un objeto estructurado usando la API nativa de OpenAI (equivalente a `04_cot/Notebooks/01_zero_shot_cot_recomendador.ipynb`).

In [4]:
prompt_cot = ChatPromptTemplate.from_messages([
    ("system", "Resuelve este problema matemático paso a paso y devuélvelo obligatoriamente en JSON con 'thoughts' (lista de str) y 'respuesta_final' (entero)."),
    ("user", "{acertijo}")
])

# Forzamos comportamiento JSON puro
from langchain_core.output_parsers import JsonOutputParser
llm_json = llm_mini.bind(response_format={"type": "json_object"})

cot_chain = prompt_cot | llm_json | JsonOutputParser()

print("---- Output Chain of Thought en formato JSON determinístico ----")
try:
    res_cot = cot_chain.invoke({"acertijo": "Tengo 10 contenedores EC2 activos. Dos fallan, y levanto otros 5 en AutoScaling. ¿Cuántos totales limpios y estables me quedan?"})
    import json
    print(json.dumps(res_cot, indent=2, ensure_ascii=False))
except Exception as e:
    print("[WARN] Requieres API KEY de OpenAI para correr la celda:", e)

---- Output Chain of Thought en formato JSON determinístico ----


{
  "thoughts": [
    "Comienzo con 10 contenedores EC2 activos.",
    "De esos 10, 2 fallan, por lo que quedan 10 - 2 = 8 contenedores activos.",
    "Luego, levanto 5 contenedores adicionales en AutoScaling.",
    "Por lo tanto, el total de contenedores activos ahora es 8 + 5 = 13.",
    "Finalmente, todos los contenedores levantados son limpios y estables, así que el total es 13."
  ],
  "respuesta_final": 13
}


---
## Módulo 01: Orquestación con LangGraph (`04_langchain_langgraph`)
Las cadenas LCEL fallan miserablemente si necesitamos que el LLM ejecute tareas asimétricas en paralelo, o si queremos que repita un bucle hasta que una regla condicional se cumpla (Evaluator-Optimizer o Routing).

Aquí reimplementamos el flujo interno sacado directamente de `05_routing/routing_langgraph.ipynb`: Un **StateGraph** estricto.

In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

# 1. Definición del Estado inmutable del Grafo en el que mutaremos valores
class SystemState(TypedDict):
    usuario_input: str
    ruta_decidida: str
    respuesta_maquina: str

# 2. Nodos Operativos
def nodo_clasificador(state: SystemState):
    print("[Grafo Activo] ==> Clasificando...")
    q = state["usuario_input"].lower()
    # Podría ser un LLM; para no gastar tokens, es heurístico
    if "factura" in q:
        return {"ruta_decidida": "ventas"}
    return {"ruta_decidida": "tecnico"}

def nodo_robot_ventas(state: SystemState):
    print("[Grafo Activo] ==> Ejecutando Robot de VENTAS")
    return {"respuesta_maquina": "Generando tickets de Billing API."}

def nodo_robot_tecnico(state: SystemState):
    print("[Grafo Activo] ==> Ejecutando Robot de SOPORTE")
    return {"respuesta_maquina": "Escalando log de errores en Datadog."}

# 3. Lógica del Router Edge (Devuelve un string apuntando al próximo Nodo)
def decide_next_node(state: SystemState) -> str:
    return state["ruta_decidida"]

# 4. Ensamblaje (Wiring del Routing Graph)
workflow = StateGraph(SystemState)
workflow.add_node("clasificador", nodo_clasificador)
workflow.add_node("ventas", nodo_robot_ventas)
workflow.add_node("tecnico", nodo_robot_tecnico)

workflow.set_entry_point("clasificador")

# El Routing Edge Mágico
workflow.add_conditional_edges(
    "clasificador",
    decide_next_node,
    {
        "ventas": "ventas",
        "tecnico": "tecnico"
    }
)

workflow.add_edge("ventas", END)
workflow.add_edge("tecnico", END)

# Compilación a aplicacion
app_grafo = workflow.compile()

print("\n--- [TEST] Solicitud de Cobro ---")
res1 = app_grafo.invoke({"usuario_input": "Tengo un problema con la factura de este mes"})
print("Respuesta Final:", res1["respuesta_maquina"])

print("\n--- [TEST] Falla Técnica ---")
res2 = app_grafo.invoke({"usuario_input": "Error 500 al compilar el pipeline."})
print("Respuesta Final:", res2["respuesta_maquina"])


--- [TEST] Solicitud de Cobro ---
[Grafo Activo] ==> Clasificando...
[Grafo Activo] ==> Ejecutando Robot de VENTAS
Respuesta Final: Generando tickets de Billing API.

--- [TEST] Falla Técnica ---
[Grafo Activo] ==> Clasificando...
[Grafo Activo] ==> Ejecutando Robot de SOPORTE
Respuesta Final: Escalando log de errores en Datadog.


---
## Módulo 02: Fundamentos Vectoriales, Tokens y Distancia Semántica (`01_intro` + `02_databases`)
Los agentes y los sistemas LLM fallan por alucinaciones al carecer de conocimiento propietario actualizado (Memory-Wall). Superamos esto inyectando el contexto relevante.

**1. Tokens:** Trozos lingüísticos en los que se descompone la palabra con BPE.
**2. TF-IDF y Bag Of Words:** Cuentan solapamientos de palabras exactas. Es rápido, pero pésimo interpretando sinónimos.
**3. Embeddings:** Redes Transformers (MiniLM, text-embedding-3) comprimen oraciones a arreglos/arrays N-Dimensionales. Midiendo la *Distancia Coseno* podemos encontrar que "Felino Mimoso" === "Gato tierno" matemáticamente.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("--- Búsqueda Léxica TF-IDF (`01_intro/02_rag_tfidf.ipynb`) ---")
corpus = ["Documentación de LangChain", "Tutorial de LangGraph y Orquestación", "Guía de Python"]
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)
print("Vocabulario exacto extraído:", vectorizer.get_feature_names_out())

print("\n--- Búsqueda Semántica Vectorial (`02_databases`) ---")
import numpy as np
from langchain_openai import OpenAIEmbeddings

try:
    emb_model = OpenAIEmbeddings(model="text-embedding-3-small")
    v_gato = np.array(emb_model.embed_query("El gato está durmiendo"))
    v_feli = np.array(emb_model.embed_query("Un pequeño felino descansa plenamente"))
    v_carro = np.array(emb_model.embed_query("Un motor a combustión de 4 cilindros"))
    
    def cos_sim(vA, vB):
        return np.dot(vA, vB) / (np.linalg.norm(vA) * np.linalg.norm(vB))
    
    print(f"Distancia Coseno (Textos Semánticamente Parecidos): {cos_sim(v_gato, v_feli):.4f} (CERCA A 1.0)")
    print(f"Distancia Coseno (Textos Nada que ver):         {cos_sim(v_gato, v_carro):.4f} (CERCA A 0.0)")
except Exception as e:
    print("Token no detectado.")

--- Búsqueda Léxica TF-IDF (`01_intro/02_rag_tfidf.ipynb`) ---
Vocabulario exacto extraído: ['de' 'documentación' 'guía' 'langchain' 'langgraph' 'orquestación'
 'python' 'tutorial']

--- Búsqueda Semántica Vectorial (`02_databases`) ---


Distancia Coseno (Textos Semánticamente Parecidos): 0.7218 (CERCA A 1.0)
Distancia Coseno (Textos Nada que ver):         0.2456 (CERCA A 0.0)


---
## Módulo 02: Ingesta, ChromaDB y Multi-Query/Ensemble RAG (`03_rag/02-rag-avanzado`)
El *Retrieval-Augmented Generation* Básico no basta en Producción (Bajo Recall). Para mitigarlo usamos dos estrategias vistas en `02-rag-avanzado.ipynb`:

1. **Chunking Recursive:** Cortar metódicamente Párrafos superpuestos (`RecursiveCharacterTextSplitter`).
2. **Multi-Query RAG:** Si un usuario busca mal una keyword, usamos el LLM para re-frasear su pregunta en 3 variantes lógicas.
3. **Ensemble RRF (Reciprocal Rank Fusion):** Combinar los resultados del Semántico(`Chroma`) y del Léxico(`BM25`) en lista matemática unificada. Literalmente la forma más robusta de buscar metadatos exactos (Precios, SKUs) sin perder el significado heurístico del parráfo.

In [7]:
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_text_splitters import RecursiveCharacterTextSplitter

texto_crudo = "Políticas de NovaTech Solutions. Todos los empleados de tiempo completo tienen 22 días de vacaciones. El plan de Salud es MediPlus y el seguro dental tiene un copago de $150 USD y $20 en medicinas. La sede central es en la Avenida Independencia."

splitter = RecursiveCharacterTextSplitter(chunk_size=75, chunk_overlap=15)
chunks = splitter.create_documents([texto_crudo])

print(f"[Chunking Resultante]: {len(chunks)} trozos.")
print("Trozo Muestra:", chunks[1].page_content)

# Simulando Ensemble Retreat.
bm25_searcher = BM25Retriever.from_documents(chunks)
print("\n[BM25 Búsqueda Léxica Pura exacta - 'MediPlus']: ")
try:
    print(bm25_searcher.invoke("MediPlus")[0].page_content)
except:
    pass

# Simulación Multi-Query Generator
prompt_mq = ChatPromptTemplate.from_template("Genera 3 versiones SEO alternativas de esta pregunta.\nPregunta:{question}")
mq_chain = prompt_mq | llm_mini | StrOutputParser()

print("\n[Generador Multi-Query RAG Ampliación Vocabulario]:")
try:
    print(mq_chain.invoke({"question": "Qué seguro de salud tengo?"}))
    # En prod, iteraríamos por cada una de esas 3 respuestas en la DB Vectorial
except:
    pass


[Chunking Resultante]: 4 trozos.
Trozo Muestra: completo tienen 22 días de vacaciones. El plan de Salud es MediPlus y el

[BM25 Búsqueda Léxica Pura exacta - 'MediPlus']: 
USD y $20 en medicinas. La sede central es en la Avenida Independencia.

[Generador Multi-Query RAG Ampliación Vocabulario]:


Claro, aquí tienes tres versiones SEO alternativas de la pregunta "¿Qué seguro de salud tengo?":

1. **¿Cómo puedo saber qué tipo de seguro de salud tengo?**
2. **¿Qué información necesito para identificar mi seguro de salud?**
3. **¿Cuáles son los pasos para averiguar mi seguro de salud actual?**


---
## Módulo 02 Finale: Agentic RAG Enlazando todo a Gotham (`04_batman_vector_db_orchestration`)

Al final del Módulo 02 aprendimos que RAG no tiene que ser un bloque de texto estático inyectado de golpe. **Agentic RAG** delega todo: Proveemos al Agente Base con la Herramienta (Tool) para consultar él mismo la base temporal. Él decide los parámetros.

El archivo culminante (`05_agent2agent_roles_router_batman.ipynb`) mezclaba todo: Routing, Agent Tools, Vector DB.

In [8]:
from langchain_core.tools import tool

@tool
def consultar_arsenal_batman(equipo_solicitado: str) -> str:
    """
    Consulta el sistema local de Baticueva para saber qué armamento o vehículos hay disponibles.
    """
    print(f"[SYSTEM] Query simulado interno sobre ChromaDB Vectorial con {equipo_solicitado}...")
    if "batmovil" in equipo_solicitado.lower():
        return "Base de datos indica: El Batmóvil está en mantenimiento preventivo (Rueda Izquierda)."
    return "Equipamiento disponible en perchas 4 y 5."

# Usamos bind_tools para transformar a nuestro predictivo base en un Agentic Executor
batman_agente = llm_mini.bind_tools([consultar_arsenal_batman])

try:
    # ReAct in action (Thought) 
    mensaje_batman = batman_agente.invoke("Identifica si puedo usar el Batmóvil para patrullar hoy.")
    print("\n[BATMAN LLM THOUGHT -> Decisión de Usar Herramientas]:")
    print("Calls solicitadas:", mensaje_batman.tool_calls)
except Exception as e:
    pass



[BATMAN LLM THOUGHT -> Decisión de Usar Herramientas]:
Calls solicitadas: [{'name': 'consultar_arsenal_batman', 'args': {'equipo_solicitado': 'Batmóvil'}, 'id': 'call_36A26bnk7XsU7Gz3o9Frur1r', 'type': 'tool_call'}]


## Cierre Magistral (Aprobando el Midterm del MIT)

He repasado línea a línea la estructura de absolutamente todos los ejercicios combinados (Bases de Python Pydantic -> Prompt LCEL -> Iteración CoT/ReAct -> Orquestación Condicional con StateGraph -> Búsquedas Ensemble Léxicas/Vectoriales de Base de Datos).

La teoría terminó. Su código ahora entiende contexto. Con estas primitivas integradas, el paso firme hacia los Sistemas Multi-Agente Asíncronos del **Módulo 03** los espera sin obstáculos.